# Historical prices

The `/price` endpoint returns the update that was active at a specific Unix microsecond
timestamp. This lets you replay historical feed values at arbitrary sub-second resolution.

In [ ]:
!pip install -q pyth-pandas matplotlib python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import matplotlib.pyplot as plt

from pyth_pandas import PythPandas

client = PythPandas()

## Build a range of timestamps

Pick one-minute grid points over the last hour. Pyth's `fetch_prices` accepts any
datetime-like input (ints, `pd.Timestamp`, ISO strings); internally it's converted to
microsecond-precision Unix time.

In [ ]:
end = pd.Timestamp.now(tz="UTC").floor("1min") - pd.Timedelta(minutes=1)
grid = pd.date_range(end=end, periods=60, freq="1min", tz="UTC")
grid[:5]

## Fetch each snapshot

One API call per grid point. For heavy backfills use the `AsyncPythPandas` client to
parallelize — it shares the same method signatures.

In [ ]:
rows = []
for ts in grid:
    snap = client.fetch_prices(
        timestamp=ts,
        symbols=["Crypto.BTC/USD", "Crypto.ETH/USD"],
        properties=["price", "exponent", "confidence"],
        formats=[],
    )
    snap["snapshotAt"] = ts
    rows.append(snap)

history = pd.concat(rows, ignore_index=True)
history["humanPrice"] = history["price"] * 10 ** history["exponent"]
history["humanConfidence"] = history["confidence"] * 10 ** history["exponent"]
history.head()

## Plot

Pivot by feed and overlay the series.

In [ ]:
wide = history.pivot_table(
    index="snapshotAt",
    columns="priceFeedId",
    values="humanPrice",
).sort_index()

fig, axes = plt.subplots(1, wide.shape[1], figsize=(12, 4), sharex=True)
if wide.shape[1] == 1:
    axes = [axes]
for ax, col in zip(axes, wide.columns):
    wide[col].plot(ax=ax, marker=".", linewidth=1)
    ax.set_title(f"Feed {col}")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Confidence bands

Each feed ships a `confidence` value that Pyth publishers use to express uncertainty.
It's in the same units as `price`, so you can plot a ± confidence band directly.

In [ ]:
feed = history[history["priceFeedId"] == history["priceFeedId"].iloc[0]].sort_values("snapshotAt")

plt.figure(figsize=(10, 4))
plt.plot(feed["snapshotAt"], feed["humanPrice"], linewidth=1.2, label="price")
plt.fill_between(
    feed["snapshotAt"],
    feed["humanPrice"] - feed["humanConfidence"],
    feed["humanPrice"] + feed["humanConfidence"],
    alpha=0.2,
    label="± confidence",
)
plt.title(f"Feed {feed['priceFeedId'].iloc[0]} — price ± confidence")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()